# UAE Insurance Market Historical Line-of-Business Deep Dive

### Purpose
Supports the second half of the business task: having identified the UAE 
as a focus market, what product-line trends over the past several years 
should shape entry strategy?

### Data Source
Central Bank of the UAE (CBUAE), "Annual Statistical Report for the 
Insurance Sector of the UAE" editions 2022, 2023, 2024, and 2025, 
Table 2A ("Gross Written Premium by Line of Business National & 
Foreign Companies"). Each edition reports the current and prior year; 
overlapping years were cross-checked across editions and found identical, 
confirming accuracy.

### Coverage
2021–2025, AED millions, National + Foreign companies combined.

### Notebook Structure
1. Setup & Load Data
2. Data Quality Checks
3. Currency Conversion (AED → USD)
4. Save Cleaned Output

**Input:** `data/raw/uae_gwp_by_line_2021_2025.csv`

**Output:** `data/cleaned/uae_gwp_by_line_aed.csv`, `data/cleaned/uae_gwp_by_line_usd.csv`

### 1. Setup & Load Data

In [17]:
import pandas as pd

UAE_PATH = "data/raw/uae_gwp_by_line_2021_2025.csv"
uae_lines = pd.read_csv(UAE_PATH)

assert uae_lines.shape == (14, 6), f"Unexpected shape: {uae_lines.shape}"

uae_lines

,line_of_business,2021,2022,2023,2024,2025
0,Fire,3342.522,4303.331,4529.643,6233.255,6887.188
1,Marine & Aviation,1467.499,1819.333,2215.793,2317.571,2258.554
2,Motor & Transportation,4731.141,4758.861,5886.563,8051.645,10075.906
3,Engineering Construction & Energy,2476.823,3222.672,3317.015,4102.537,4284.082
4,Other,3461.121,3706.961,4816.965,5588.521,6507.277
5,Reinsurance - Property & Liability,0.000,5.711,5.989,5.754,5.654
6,Total Property & Liability,15479.105,17816.869,20771.969,26299.283,30018.662
7,Total Health Insurance,19868.130,21741.970,25897.043,31317.757,36422.697
8,Group Life,832.607,912.909,1020.277,1121.586,1448.263
9,Group Credit Life,947.495,828.444,561.185,630.315,720.536


### 2. Data Quality Checks

Since this dataset includes both individual line items and their stated 
subtotals/total, we can verify internal consistency directly the 
individual lines should sum to their reported totals for every year.

In [18]:
print("Shape:", uae_lines.shape)
print("\nMissing values:\n", uae_lines.isnull().sum())
print("\nDuplicate line items:", uae_lines["line_of_business"].duplicated().sum())

Shape: (14, 6)

Missing values:
 line_of_business    0
2021                0
2022                0
2023                0
2024                0
2025                0
dtype: int64

Duplicate line items: 0


In [19]:
uae_lines[["2021","2022","2023","2024","2025"]].describe()

,2021,2022,2023,2024,2025
count,14.000000,14.000000,14.000000,14.000000,14.000000
mean,8077.354143,8571.240500,9588.320286,11715.325000,13435.647286
std,11996.758097,12880.318391,14756.154306,18055.049024,20806.808452
min,0.000000,5.711000,5.989000,5.754000,5.654000
25%,1077.496000,1139.515000,1319.156000,1420.582250,1650.835750
50%,3401.821500,4005.146000,4673.304000,5644.291000,6354.776000
75%,8498.153750,7215.627000,6503.264000,7912.166000,9656.736500
max,44317.029000,47246.445000,53377.843000,65110.769000,74840.586000


In [20]:
pl_components = ["Fire", "Marine & Aviation", "Motor & Transportation",
                  "Engineering Construction & Energy", "Other",
                  "Reinsurance - Property & Liability"]

pl_check = uae_lines[uae_lines["line_of_business"].isin(pl_components)][["2021","2022","2023","2024","2025"]].sum()
pl_reported = uae_lines[uae_lines["line_of_business"] == "Total Property & Liability"][["2021","2022","2023","2024","2025"]].iloc[0]

print("Calculated P&L sum:\n", pl_check)
print("\nReported P&L total:\n", pl_reported)
print("\nDifference:\n", (pl_check - pl_reported).round(3))

Calculated P&L sum:
 2021    15479.106
2022    17816.869
2023    20771.968
2024    26299.283
2025    30018.661
dtype: float64

Reported P&L total:
 2021    15479.105
2022    17816.869
2023    20771.969
2024    26299.283
2025    30018.662
Name: 6, dtype: float64

Difference:
 2021    0.001
2022    0.000
2023   -0.001
2024    0.000
2025   -0.001
dtype: float64


In [21]:
life_components = ["Group Life", "Group Credit Life", "Individual Life", "Annuities & Fund Accumulation"]

life_check = uae_lines[uae_lines["line_of_business"].isin(life_components)][["2021","2022","2023","2024","2025"]].sum()
life_reported = uae_lines[uae_lines["line_of_business"] == "Total Life"][["2021","2022","2023","2024","2025"]].iloc[0]

print("Calculated Life sum:\n", life_check)
print("\nReported Life total:\n", life_reported)
print("\nDifference:\n", (life_check - life_reported).round(3))

Calculated Life sum:
 2021    8969.794
2022    7687.607
2023    6708.830
2024    7493.729
2025    8399.228
dtype: float64

Reported Life total:
 2021    8969.794
2022    7687.607
2023    6708.831
2024    7493.729
2025    8399.228
Name: 12, dtype: float64

Difference:
 2021    0.000
2022    0.000
2023   -0.001
2024   -0.000
2025    0.000
dtype: float64


In [22]:
grand_check = pl_reported + life_reported + uae_lines[uae_lines["line_of_business"] == "Total Health Insurance"][["2021","2022","2023","2024","2025"]].iloc[0]
grand_reported = uae_lines[uae_lines["line_of_business"] == "Total All Lines"][["2021","2022","2023","2024","2025"]].iloc[0]

print("Calculated Grand Total:\n", grand_check)
print("\nReported Grand Total:\n", grand_reported)
print("\nDifference:\n", (grand_check - grand_reported).round(3))

Calculated Grand Total:
 2021    44317.029
2022    47246.446
2023    53377.843
2024    65110.769
2025    74840.587
dtype: float64

Reported Grand Total:
 2021    44317.029
2022    47246.445
2023    53377.843
2024    65110.769
2025    74840.586
Name: 13, dtype: float64

Difference:
 2021   -0.000
2022    0.001
2023    0.000
2024    0.000
2025    0.001
dtype: float64


### Finding: Internal Consistency Verified

All three checks (Property & Liability subtotal, Life subtotal, and Grand 
Total) match the report's stated totals within ±0.001 AED million across 
all five years negligible floating-point rounding from the thousands-to-
millions conversion, not a data discrepancy. This confirms the manually 
transcribed line-item data is accurate and internally consistent with the 
official regulator totals.

**Decision:** no adjustments needed; data is validated as-is.

### 3. Currency Conversion (AED → USD)

The AED has been pegged at a fixed rate to the US Dollar since 1997 
(1 USD = 3.6725 AED), so conversion introduces no exchange-rate-year 
ambiguity, unlike floating currencies.

In [23]:
AED_TO_USD = 1 / 3.6725

year_cols = ["2021", "2022", "2023", "2024", "2025"]
uae_lines_usd = uae_lines.copy()
uae_lines_usd[year_cols] = (uae_lines[year_cols] * AED_TO_USD).round(2)

uae_lines_usd

,line_of_business,2021,2022,2023,2024,2025
0,Fire,910.15,1171.77,1233.39,1697.28,1875.34
1,Marine & Aviation,399.59,495.39,603.35,631.06,614.99
2,Motor & Transportation,1288.26,1295.81,1602.88,2192.42,2743.61
3,Engineering Construction & Energy,674.42,877.51,903.20,1117.10,1166.53
4,Other,942.44,1009.38,1311.63,1521.72,1771.89
5,Reinsurance - Property & Liability,0.00,1.56,1.63,1.57,1.54
6,Total Property & Liability,4214.87,4851.43,5656.08,7161.14,8173.90
7,Total Health Insurance,5409.97,5920.21,7051.61,8527.64,9917.68
8,Group Life,226.71,248.58,277.82,305.40,394.35
9,Group Credit Life,258.00,225.58,152.81,171.63,196.20


### Data Dictionary `uae_gwp_by_line_aed.csv` / `uae_gwp_by_line_usd.csv`

| Column | Type | Description | Unit |
|---|---|---|---|
| line_of_business | text | Insurance line/category, or subtotal/total row | |
| 2021–2025 | float | Gross Written Premium for that line, by year | AED millions (aed file) / USD millions (usd file) |

Rows include both individual lines (Fire, Motor & Transportation, etc.) 
and three subtotals (Total Property & Liability, Total Health Insurance, 
Total Life) plus one grand total (Total All Lines) filter accordingly 
depending on whether individual-line or aggregate analysis is needed.

### Cross-Source Validation

Converting the CBUAE regulator data to USD gives a 2025 total of USD 20.38B, closely 
matching the Alpen Capital report's independent 2025E estimate of USD 20.5B for the 
UAE (used in `gcc_merged.csv`) a 0.6% difference. This cross-validates both sources and confirms the two sources agree 
closely on total UAE market size.

**Life vs. non-life split does not reconcile.** Alpen Capital reports UAE's life share 
at 17.1% of total GWP (`life_share_pct` in `gcc_insurance_2025.csv`). Recalculating 
the same ratio from CBUAE's granular data (`Total Life` ÷ `Total All Lines`, 2025) 
gives 11.2% — a 6-percentage-point gap, roughly a third lower than Alpen's figure. 
The two sources agree closely on total market size but not on this internal split, 
and the cause isn't identifiable from either source's publicly available 
documentation. Because this project uses Alpen's regional `life_share_pct` for 
cross-GCC comparisons and CBUAE's line-level data for the UAE deep dive, any finding 
that leans on both together should be read with this discrepancy in mind.

### 4. Save Cleaned Output

In [24]:
import os
os.makedirs("data/cleaned", exist_ok=True)

uae_lines.to_csv("data/cleaned/uae_gwp_by_line_aed.csv", index=False)
uae_lines_usd.to_csv("data/cleaned/uae_gwp_by_line_usd.csv", index=False)

print("Saved uae_gwp_by_line_aed.csv:", uae_lines.shape)
print("Saved uae_gwp_by_line_usd.csv:", uae_lines_usd.shape)

Saved uae_gwp_by_line_aed.csv: (14, 6)
Saved uae_gwp_by_line_usd.csv: (14, 6)


### Known Limitations

- **Health Insurance has no sub-line breakdown** in the source tables 
  it's reported as a single total, unlike Property & Liability and Life, 
  which break into individual components. Any health-specific trend 
  analysis is limited to the aggregate line.
- **"Other" (within Property & Liability) is an unspecified residual 
 category** in the source report its composition isn't detailed, so 
  its growth can't be attributed to specific sub-lines.
- **Currency conversion uses a fixed peg** (1 USD = 3.6725 AED); this is 
  exact and doesn't introduce approximation, unlike floating-currency 
  conversions elsewhere in this project.

### Summary UAE Data Collection & Cleaning Complete

Two cleaned datasets produced, both covering 2021–2025 UAE GWP by line of 
business (14 rows: 10 individual lines, 3 subtotals, 1 grand total):
- `data/cleaned/uae_gwp_by_line_aed.csv` original currency (AED millions)
- `data/cleaned/uae_gwp_by_line_usd.csv` converted at the fixed AED/USD peg

All figures validated: internal subtotals reconcile to within ±0.001 
(rounding only), and independently cross-validated against the Alpen 
Capital GCC report (0.6% difference on UAE 2025 total).

**Next notebook:** `04_analysis.ipynb` exploratory analysis and 
visualizations combining `gcc_merged.csv` and both UAE datasets.